# Day 07 — Summarization, JSON & Evaluation Modes

**Module 2 · The Metric Toolkit**

Today we explore three useful evaluation patterns:

1. **Summarization** — evaluate a summary without a golden summary.
2. **JSON correctness** — validate structured output against a schema.
3. **Evaluation modes** — understand score-only and flaky metrics.

> **Core idea:** Not every evaluation needs an expected answer, and not every metric needs to act as a pass/fail gate.

## 1. Setup

We will use OpenAI's `gpt-4o-mini` as our judge model.

We also configure DeepEval's concurrency settings via `AsyncConfig`.

In [11]:
import os

from dotenv import load_dotenv
from deepeval.models import OpenAIModel
from deepeval.evaluate import AsyncConfig

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4o-mini",
    temperature=0,
)

async_config = AsyncConfig(
    max_concurrent=4
)

print("Judge:", judge.get_model_name())
print("Max concurrent evaluations:", async_config.max_concurrent)

Judge: gpt-4o-mini
Max concurrent evaluations: 4


## 2. Summarization Without a Golden Answer

Sometimes we want to evaluate a summary, but we do not have a manually written reference summary.

Instead, we have:

```text
Original Text
      ↓
Generated Summary
      ↓
Summarization Metric
      ↓
Score + Reason

In [12]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

article = (
    "Retrieval-Augmented Generation grounds an LLM's answer in external documents. "
    "At query time the system retrieves relevant chunks and passes them to the model "
    "as context. This can reduce hallucination and allows knowledge to be updated "
    "without retraining the model."
)

summary_cases = [
    LLMTestCase(
        input=article,
        actual_output=(
            "RAG grounds LLM answers in retrieved documents, which can reduce "
            "hallucination and allow knowledge updates without retraining."
        ),
    ),
    LLMTestCase(
        input=article,
        actual_output=(
            "RAG is a technique that makes AI models faster and guarantees "
            "that their answers are always correct."
        ),
    ),
]

## 3. Run the Summarization Metric

In [13]:
from deepeval import evaluate

summarization = SummarizationMetric(
    model=judge,
    threshold=0.5,
)

results = evaluate(
    test_cases=summary_cases,
    metrics=[summarization],
    async_config=async_config,
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(f"\n{result.name}")
    print(f"Score:   {metric_result.score:.2f}")
    print(f"Success: {metric_result.success}")
    print(f"Reason:  {metric_result.reason}")

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            Retrieval-Augmented Generation grounds an LLM's answer in external documents. At       │
│  │                       query time the system retrieves relevant chunks and passes them to the model as        │
│  │                       context. This can reduce hallucination and allows knowledge to be updated without      │
│  │                       retraining the model.                                                                  │
│  │     Actual Output:    RAG is a technique that makes AI models faster and guarantees that their answers       │
│  │                       are always correct.                                                                    │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric        ┃ Score ┃ Threshold ┃ Reason                                                       │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Summarization │ 0.00  │ 0.50      │ The score is 0.00 because the summary contains a             │
│              │               │       │           │ significant contradiction regarding the accuracy of AI       │
│              │               │       │           │ model answers, claiming that RAG guarantees correctness,     │
│              │               │       │           │ which is not supported by the original text. Additionally,   │
│              │               │       │           │ it introduces extra information about RAG making AI models   │
│              │               │       │           │ faster, which is not mentioned in the original text.         │
│              │               │       │           │ Furthermore, the summary fails to address several            │
│              │               │       │           │ questions that the original text can answer, indicating a    │
│              │               │       │           │ lack of completeness and accuracy.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                  ┃ Average Score          ┃ Pass Rate                                      ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Summarization           │ 0.40                   │ 50.00

⚠ WARNING: No hyperparameters logged.
» ]8;id=153330;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 10.21s | token cost: 0.0013403999999999998 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


test_case_1
Score:   0.00
Success: False
Reason:  The score is 0.00 because the summary contains a significant contradiction regarding the accuracy of AI model answers, claiming that RAG guarantees correctness, which is not supported by the original text. Additionally, it introduces extra information about RAG making AI models faster, which is not mentioned in the original text. Furthermore, the summary fails to address several questions that the original text can answer, indicating a lack of completeness and accuracy.

test_case_0
Score:   0.80
Success: True
Reason:  The score is 0.80 because the summary accurately reflects the main points of the original text without contradictions or extra information, but it fails to address a specific question that the original text can answer.


## 4. Evaluating Structured JSON Output

LLM applications often need to return structured data.

For example:

```json
{
    "answer": "...",
    "confidence": 0.95
}

In [14]:
from pydantic import BaseModel
from deepeval.metrics import JsonCorrectnessMetric


class Answer(BaseModel):
    answer: str
    confidence: float


json_cases = [
    LLMTestCase(
        input="What is RAG?",
        actual_output=(
            '{"answer": "Retrieval-Augmented Generation", "confidence": 0.95}'
        ),
    ),
    LLMTestCase(
        input="What is RAG?",
        actual_output=(
            '{"answer": "Retrieval-Augmented Generation"}'
        ),
    ),
]

json_metric = JsonCorrectnessMetric(
    expected_schema=Answer,
    model=judge,
    threshold=0.5,
)

## 5. Run the JSON Check

In [15]:
results = evaluate(
    test_cases=json_cases,
    metrics=[json_metric],
    async_config=async_config,
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(f"\n{result.name}")
    print(f"Score:   {metric_result.score:.2f}")
    print(f"Success: {metric_result.success}")
    print(f"Reason:  {metric_result.reason}")

✨ You're running DeepEval's latest Json Correctness Metric! (using gpt-4o-mini, strict=True, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What is RAG?                                                                           │
│  │     Actual Output:    {"answer": "Retrieval-Augmented Generation"}                                           │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Json Correctness │ 0.00  │ 1.00      │ The generated Json is not valid because it is missing     │
│              │                  │       │           │ the 'confidence' property, which is required by the       │
│              │                  │       │           │ Expected Json Schema.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Json Correctness           │ 0.50                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=218254;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.5s | token cost: 7.049999999999999e-05 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


test_case_0
Score:   1.00
Success: True
Reason:  The generated Json matches and is syntactically correct to the expected schema.

test_case_1
Score:   0.00
Success: False
Reason:  The generated Json is not valid because it is missing the 'confidence' property, which is required by the Expected Json Schema.


## 6. Score-Only Mode

Not every metric needs to decide whether a test passes or fails.

Setting:

```python
threshold=None

In [16]:
primary_metric = SummarizationMetric(
    model=judge,
    threshold=0.5,
)

score_only_metric = SummarizationMetric(
    model=judge,
    threshold=None,
)

results = evaluate(
    test_cases=summary_cases,
    metrics=[primary_metric, score_only_metric],
    async_config=async_config,
)

for result in results.test_results:
    print(f"\n{result.name}")

    for metric_result in result.metrics_data:
        print(
            f"{metric_result.name}: "
            f"score={metric_result.score:.2f}, "
            f"success={metric_result.success}"
        )

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            Retrieval-Augmented Generation grounds an LLM's answer in external documents. At       │
│  │                       query time the system retrieves relevant chunks and passes them to the model as        │
│  │                       context. This can reduce hallucination and allows knowledge to be updated without      │
│  │                       retraining the model.                                                                  │
│  │     Actual Output:    RAG is a technique that makes AI models faster and guarantees that their answers       │
│  │                       are always correct.                                                                    │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric        ┃ Score ┃ Threshold ┃ Reason                                                       │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Summarization │ 0.00  │ 0.50      │ The score is 0.00 because the summary contains significant   │
│              │               │       │           │ contradictions to the original text, such as incorrectly     │
│              │               │       │           │ stating that RAG makes AI models faster and guarantees       │
│              │               │       │           │ correct answers, which the original text does not support.   │
│              │               │       │           │ Additionally, the summary fails to address several key       │
│              │               │       │           │ questions that the original text can answer.                 │
│        NONE  │ Summarization │ 0.00  │ N/A       │ The score is 0.00 because the summary contains significant   │
│              │               │       │           │ contradictions to the original text, such as incorrectly     │
│              │               │       │           │ stating that RAG makes AI models faster and guarantees       │
│              │               │       │           │ correct answers, which are not supported by the original     │
│              │               │       │           │ content. Additionally, the summary fails to address          │
│              │               │       │           │ several key questions that the original text can answer.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                          

⚠ WARNING: No hyperparameters logged.
» ]8;id=730907;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 8.54s | token cost: 0.002643 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


test_case_0
Summarization: score=0.80, success=True
Summarization: score=0.80, success=None

test_case_1
Summarization: score=0.00, success=False
Summarization: score=0.00, success=None


## 7. Score vs. Verdict

Remember the difference:

```text
Score
  ↓
How good is the output?

Threshold
  ↓
What score is acceptable?

Verdict
  ↓
PASS / FAIL

## 8. Flaky Metrics

Sometimes a metric is useful but not reliable enough to block an evaluation.

Setting:

```python
flaky=True
```

In [17]:
# Primary metric: determines PASS / FAIL
primary_metric = SummarizationMetric(
    model=judge,
    threshold=0.5,
)

# Flaky metric: score, but never fail the case
flaky_metric = SummarizationMetric(
    model=judge,
    threshold=0.5,
    flaky=True,
)

results = evaluate(
    test_cases=summary_cases,
    metrics=[primary_metric, flaky_metric],
    async_config=async_config,
)

for result in results.test_results:
    print(f"\n{result.name}")

    for metric_result in result.metrics_data:
        print(
            f"  {metric_result.name}: "
            f"score={metric_result.score:.2f}, "
            f"success={metric_result.success}"
        )

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            Retrieval-Augmented Generation grounds an LLM's answer in external documents. At       │
│  │                       query time the system retrieves relevant chunks and passes them to the model as        │
│  │                       context. This can reduce hallucination and allows knowledge to be updated without      │
│  │                       retraining the model.                                                                  │
│  │     Actual Output:    RAG is a technique that makes AI models faster and guarantees that their answers       │
│  │                       are always correct.                                                                    │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric        ┃ Score ┃ Threshold ┃ Reason                                                       │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Summarization │ 0.00  │ 0.50      │ The score is 0.00 because the summary contains a             │
│              │               │       │           │ significant contradiction regarding the accuracy of AI       │
│              │               │       │           │ model answers, claiming that RAG guarantees correctness,     │
│              │               │       │           │ which is not supported by the original text. Additionally,   │
│              │               │       │           │ it introduces extra information about RAG making AI models   │
│              │               │       │           │ faster, which is not mentioned in the original text.         │
│              │               │       │           │ Furthermore, the summary fails to address several            │
│              │               │       │           │ questions that the original text can answer, indicating a    │
│              │               │       │           │ lack of completeness and accuracy.                           │
│        FAIL  │ Summarization │ 0.00  │ 0.50      │ The score is 0.00 because the summary contains significant   │
│              │               │       │           │ contradictions to the original text, such as incorrect       │
│              │               │       │           │ claims about RAG making AI models faster and guaranteeing    │
│              │               │       │           │ correct answers, which are not supported by the original     │
│              │               │       │           │ content.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭──────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=947024;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.58s | token cost: 0.0026550000000000002 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


test_case_0
  Summarization: score=0.80, success=True
  Summarization: score=0.80, success=True

test_case_1
  Summarization: score=0.00, success=False
  Summarization: score=0.00, success=False


# Day 07 — Key Takeaways

Today we learned:

### Summarization
Evaluate a generated summary against its source text without requiring a golden summary.

### JSON Correctness
Validate structured LLM output against a Pydantic schema.

### Score-only
Use `threshold=None` when we want scores without a pass/fail gate (run alongside a primary gating metric in `evaluate()`).

### Flaky
Use `flaky=True` when we want to measure a metric without allowing its failure to determine the test result.

And throughout:

```text
OpenAI Judge (gpt-4o-mini)
    ↓
max_concurrent = 4
```